[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/brilliantbeaver/alexpose/blob/main/penny/gavd3/10_latent_world_model_forward_prediction.ipynb)

# 10. From infilling to a causal gait world model

Train a causal S-JEPA variant that forecasts the latent representation of the next 16 frames from the previous 48, then measure per-horizon error against a phase-conditioned baseline and an out-of-distribution check on abnormal gait.

**Research use only.** This tutorial does not diagnose a person or validate a clinical device.

**Run it:** locally, use `uv sync` then `uv run jupyter lab` from this folder. In Colab, use the badge and run the setup cell. Restart the kernel after changing `penny/gavd3/.env`.

**Keep the walk visible:** notebook 03 draws the joint-time token grid and notebook 04 explains the EMA teacher and the latent objective. Revisit them whenever a forecast number looks surprising.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/brilliantbeaver/alexpose.git"

if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "numpy", "pandas", "scipy", "scikit-learn", "matplotlib",
        "seaborn", "torch", "tqdm", "python-dotenv", "yt-dlp[default]",
        "opencv-python-headless", "mediapipe<1", "joblib", "pyarrow",
    ])
    clone_dir = Path("/content/alexpose")
    if not (clone_dir / ".git").exists():
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(clone_dir)])
    os.chdir(clone_dir)


def find_project_root(start=None):
    env_root = os.getenv("ALEXPOSE_ROOT")
    if env_root:
        candidate = Path(env_root).expanduser().resolve()
        if (candidate / ".git").exists() and (candidate / "data" / "gavd").exists():
            return candidate
        print(f"Ignoring invalid ALEXPOSE_ROOT: {candidate}")
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists() and (candidate / "data" / "gavd").exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root()
TUTORIAL_DIR = PROJECT_ROOT / "penny" / "gavd3"

try:
    from dotenv import load_dotenv
    load_dotenv(TUTORIAL_DIR / ".env", override=False)
    load_dotenv(PROJECT_ROOT / ".env", override=False)
except Exception:
    pass

MODE = os.getenv("GAVD3_MODE", "smoke").strip().lower()
if MODE not in {"smoke", "real"}:
    raise ValueError("GAVD3_MODE must be smoke or real")
if MODE == "smoke":
    print(
        "SMOKE MODE: hand-authored motions test code paths only. "
        "They have no pathophysiological or clinical validity."
    )

PREFERRED_ROOT = Path(
    os.getenv(
        "GAVD4_ROOT",
        "/Users/pmui/vaults/worldmodels/gait/skeleton-jepa/gavd4",
    )
).expanduser()

requested_data = os.getenv("GAVD4_DATA_DIR") or os.getenv("GAVD_DATA_GAVD_DIR")
if requested_data and Path(requested_data).expanduser().exists():
    DATA_GAVD_DIR = Path(requested_data).expanduser()
elif requested_data:
    print(f"Ignoring missing GAVD CSV path: {Path(requested_data).expanduser()}")
    if (PREFERRED_ROOT / "data-gavd").exists():
        DATA_GAVD_DIR = PREFERRED_ROOT / "data-gavd"
    else:
        DATA_GAVD_DIR = PROJECT_ROOT / "data" / "gavd"
elif (PREFERRED_ROOT / "data-gavd").exists():
    DATA_GAVD_DIR = PREFERRED_ROOT / "data-gavd"
else:
    DATA_GAVD_DIR = PROJECT_ROOT / "data" / "gavd"

requested_youtube = os.getenv("GAVD4_YOUTUBE_DIR") or os.getenv("GAVD_YOUTUBE_DIR")
if requested_youtube:
    YOUTUBE_DIR = Path(requested_youtube).expanduser()
elif PREFERRED_ROOT.exists():
    YOUTUBE_DIR = PREFERRED_ROOT / "youtube"
else:
    YOUTUBE_DIR = PROJECT_ROOT / "penny" / "gavd3" / "work" / "youtube"

CACHE_DIR = Path(
    os.getenv("GAVD3_CACHE_DIR", TUTORIAL_DIR / "work" / "cache")
).expanduser()
ARTIFACT_ROOT = Path(
    os.getenv("GAVD3_ARTIFACT_DIR", TUTORIAL_DIR / "work" / "artifacts")
).expanduser()
ARTIFACT_DIR = ARTIFACT_ROOT / MODE
POSE_DIR = ARTIFACT_DIR / "poses"

for folder in [CACHE_DIR, ARTIFACT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

os.environ.setdefault("MPLCONFIGDIR", str(CACHE_DIR / "matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(CACHE_DIR / "xdg-cache"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["XDG_CACHE_HOME"]).mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
from IPython.display import SVG, display


def show_tutorial_svg(filename):
    '''Render a repository SVG reliably in local Jupyter and Colab.'''
    path = TUTORIAL_DIR / "images" / filename
    if not path.exists():
        raise FileNotFoundError(
            f"Missing tutorial figure {path}. Clone the full alexpose repository."
        )
    display(SVG(filename=str(path)))

print(f"mode: {MODE}")
print(f"project: {PROJECT_ROOT}")
print(f"GAVD CSVs: {DATA_GAVD_DIR}")
print(f"YouTube cache: {YOUTUBE_DIR}")
print(f"artifacts: {ARTIFACT_DIR}")


## A world model predicts the next latent state

In his 2022 essay "A Path Towards Autonomous Machine Intelligence", Yann LeCun argues that an intelligent agent needs a world model: an internal, learned model of how its environment behaves, so the agent can simulate what happens next before it acts. Two design rules follow. First, prediction happens in a representation space, not in raw pixels or raw joint angles; reconstructing every coordinate is wasteful and brittle. Second, the model should be predictive and mostly causal, because a simulation only helps if it runs forward in time.

S-JEPA already realizes the first rule. Its view encoder, EMA teacher, and predictor live entirely in a learned latent space, and notebook 04 trained them with the latent cross-entropy objective. What S-JEPA has not yet done is honor the second rule. Notebooks 03 to 06 hide target patches in the middle of each clip. The view encoder sees context on both sides of every target, so the predictor is really doing infilling: spatial and joint statistics plus a little local motion are enough, and looking backward through time is always allowed.

A forecast flips that geometry. Hide only the trailing part of the clip, keep the earlier part, and require the model to produce the missing future from the past alone. Now the answer is not present anywhere in the input; the predictor must lean on learned gait dynamics. That is exactly the knowledge a world model should store, which is why causal masking matters. Notebook 00 warned "do not call this future prediction" about the infilling objective. This notebook finally earns the name, at the price of a documented limitation that the next section states plainly.

In [ ]:
BLAZEPOSE_33 = [
    "NOSE", "LEFT_EYE_INNER", "LEFT_EYE", "LEFT_EYE_OUTER",
    "RIGHT_EYE_INNER", "RIGHT_EYE", "RIGHT_EYE_OUTER", "LEFT_EAR",
    "RIGHT_EAR", "MOUTH_LEFT", "MOUTH_RIGHT", "LEFT_SHOULDER",
    "RIGHT_SHOULDER", "LEFT_ELBOW", "RIGHT_ELBOW", "LEFT_WRIST",
    "RIGHT_WRIST", "LEFT_PINKY", "RIGHT_PINKY", "LEFT_INDEX",
    "RIGHT_INDEX", "LEFT_THUMB", "RIGHT_THUMB", "LEFT_HIP",
    "RIGHT_HIP", "LEFT_KNEE", "RIGHT_KNEE", "LEFT_ANKLE",
    "RIGHT_ANKLE", "LEFT_HEEL", "RIGHT_HEEL", "LEFT_FOOT_INDEX",
    "RIGHT_FOOT_INDEX",
]
MASK_KEYPOINTS = [11, 12, 23, 24, 25, 26, 27, 28, 31, 32]
assert [BLAZEPOSE_33[i] for i in MASK_KEYPOINTS] == [
    "LEFT_SHOULDER", "RIGHT_SHOULDER", "LEFT_HIP", "RIGHT_HIP",
    "LEFT_KNEE", "RIGHT_KNEE", "LEFT_ANKLE", "RIGHT_ANKLE",
    "LEFT_FOOT_INDEX", "RIGHT_FOOT_INDEX",
]



CONDITIONS = ["normal", "parkinsons", "stroke", "cerebralpalsy", "myopathic"]




In [ ]:
def synthetic_gait_sequence(condition="normal", frames=64, seed=0):
    '''Create a code-path fixture, not a physiological disease simulation.'''
    rng = np.random.default_rng(seed)
    phase = np.linspace(0.0, 4.0 * np.pi, frames, endpoint=False)
    seq = np.zeros((frames, 33, 4), dtype=np.float32)
    seq[..., 3] = 1.0
    base = {
        11: (0.42, 0.28), 12: (0.58, 0.28),
        23: (0.45, 0.52), 24: (0.55, 0.52),
        25: (0.44, 0.70), 26: (0.56, 0.70),
        27: (0.43, 0.89), 28: (0.57, 0.89),
        29: (0.42, 0.92), 30: (0.58, 0.92),
        31: (0.39, 0.94), 32: (0.61, 0.94),
    }
    for joint, (x, y) in base.items():
        seq[:, joint, 0] = x
        seq[:, joint, 1] = y
    amplitude = 0.045
    lift = 0.025
    if condition == "parkinsons":
        amplitude *= 0.45
        lift *= 0.45
    if condition == "myopathic":
        seq[:, [11, 12], 0] += 0.03 * np.sin(phase)[:, None]
        seq[:, [23, 24], 0] += 0.018 * np.sin(phase)[:, None]
    for joint, knee, foot, offset in [(27, 25, 31, 0.0), (28, 26, 32, np.pi)]:
        wave = np.sin(phase + offset)
        if condition == "stroke" and joint == 27:
            wave = 0.35 * wave
        if condition == "cerebralpalsy":
            seq[:, knee, 1] -= 0.045
            seq[:, joint, 1] -= 0.02
        seq[:, joint, 0] += amplitude * wave
        seq[:, knee, 0] += 0.4 * amplitude * wave
        seq[:, foot, 0] += amplitude * wave
        seq[:, joint, 1] -= lift * np.maximum(wave, 0.0)
        seq[:, foot, 1] -= 0.7 * lift * np.maximum(wave, 0.0)
    seq[..., :3] += rng.normal(0.0, 0.0025, seq[..., :3].shape)
    return seq


def synthetic_corpus(conditions=None, n_per_condition=10, frames=64, seed=42):
    if conditions is None:
        conditions = [
            "normal", "parkinsons", "stroke", "cerebralpalsy", "myopathic"
        ]
    records = []
    counter = 0
    for condition in conditions:
        for sample in range(n_per_condition):
            records.append({
                "condition": condition,
                "sequence_id": f"smoke_{condition}_{sample:03d}",
                "video_id": f"smoke_video_{condition}_{sample // 2:02d}",
                "sequence": synthetic_gait_sequence(
                    condition=condition,
                    frames=frames,
                    seed=seed + counter,
                ),
            })
            counter += 1
    return records




In [ ]:
def interpolate_low_visibility(sequence, threshold=0.45, max_gap=4):
    '''Fill only short internal gaps and preserve the original validity mask.

    Long gaps and sequence ends are never extrapolated. Their coordinates remain
    missing until center_and_scale converts them to an explicit zero sentinel.
    They can never become S-JEPA prediction targets.
    '''
    sequence = np.asarray(sequence, dtype=np.float32).copy()
    if sequence.ndim != 3 or sequence.shape[1:] != (33, 4):
        raise ValueError(f"Expected [T, 33, 4], received {sequence.shape}")
    visibility = np.nan_to_num(sequence[..., 3], nan=0.0)
    finite = np.isfinite(sequence[..., :3]).all(axis=-1)
    valid = (visibility >= threshold) & finite
    filled = valid.copy()
    for joint in range(33):
        observed = np.flatnonzero(valid[:, joint])
        for left, right in zip(observed[:-1], observed[1:]):
            gap = int(right - left - 1)
            if not 0 < gap <= max_gap:
                continue
            fraction = (
                np.arange(1, gap + 1, dtype=np.float32) / (gap + 1)
            )[:, None]
            sequence[left + 1:right, joint, :3] = (
                sequence[left, joint, :3][None, :] * (1.0 - fraction)
                + sequence[right, joint, :3][None, :] * fraction
            )
            filled[left + 1:right, joint] = True
        sequence[~filled[:, joint], joint, :3] = np.nan
    sequence[..., 3] = visibility
    return sequence, valid


def center_and_scale(sequence, eps=1e-6):
    sequence = np.asarray(sequence, dtype=np.float32).copy()
    xyz = sequence[..., :3]
    left_hip, right_hip = xyz[:, 23], xyz[:, 24]
    left_ok = np.isfinite(left_hip).all(axis=1)
    right_ok = np.isfinite(right_hip).all(axis=1)
    pelvis = np.full((len(xyz), 3), np.nan, dtype=np.float32)
    pelvis[left_ok & right_ok] = 0.5 * (
        left_hip[left_ok & right_ok] + right_hip[left_ok & right_ok]
    )
    pelvis[left_ok & ~right_ok] = left_hip[left_ok & ~right_ok]
    pelvis[right_ok & ~left_ok] = right_hip[right_ok & ~left_ok]
    pelvis_ok = np.isfinite(pelvis).all(axis=1)
    fallback = np.median(pelvis[pelvis_ok], axis=0) if pelvis_ok.any() else np.zeros(3)
    pelvis[~np.isfinite(pelvis).all(axis=1)] = fallback
    xyz = xyz - pelvis[:, None, :]
    shoulder_width = np.linalg.norm(xyz[:, 11, :2] - xyz[:, 12, :2], axis=-1)
    hip_width = np.linalg.norm(xyz[:, 23, :2] - xyz[:, 24, :2], axis=-1)
    body_scale = np.nanmedian(np.maximum(shoulder_width, hip_width))
    if not np.isfinite(body_scale) or body_scale < eps:
        body_scale = 1.0
    sequence[..., :3] = np.nan_to_num(
        xyz / body_scale, nan=0.0, posinf=0.0, neginf=0.0
    )
    return np.nan_to_num(sequence, nan=0.0, posinf=0.0, neginf=0.0)


def temporal_resize(array, frames):
    array = np.asarray(array)
    if len(array) == frames:
        return array.copy()
    if len(array) < 2:
        return np.repeat(array, frames, axis=0)
    old_t = np.linspace(0.0, 1.0, len(array))
    new_t = np.linspace(0.0, 1.0, frames)
    flat = array.reshape(len(array), -1)
    resized = np.stack(
        [np.interp(new_t, old_t, flat[:, i]) for i in range(flat.shape[1])],
        axis=1,
    )
    return resized.reshape(frames, *array.shape[1:]).astype(array.dtype)


def prepare_sequence(
    sequence,
    frames=64,
    visibility_threshold=0.45,
    max_gap=4,
):
    cleaned, valid = interpolate_low_visibility(
        sequence, visibility_threshold, max_gap=max_gap
    )
    cleaned = center_and_scale(cleaned)
    cleaned = temporal_resize(cleaned, frames)
    valid = temporal_resize(valid.astype(np.float32), frames) >= 0.5
    return cleaned[..., :3].astype(np.float32), valid.astype(bool)




In [ ]:
def uniform_neurologic_mask(valid_patch, mask_fraction=0.60, seed=None):
    """Sample eligible joint-time tokens uniformly, without motion scores.

    valid_patch has shape [B, S, V]. True means that a patch can be a target.
    The returned mask has the same shape. True means hidden from the view encoder.
    """
    valid_patch = np.asarray(valid_patch, dtype=bool)
    if valid_patch.ndim != 3 or valid_patch.shape[2] != 33:
        raise ValueError(f"Expected [B, S, 33], received {valid_patch.shape}")
    if not 0.0 < mask_fraction < 1.0:
        raise ValueError("mask_fraction must be between 0 and 1")
    rng = np.random.default_rng(seed)
    eligible_joint = np.zeros(33, dtype=bool)
    eligible_joint[MASK_KEYPOINTS] = True
    eligible = valid_patch & eligible_joint[None, None, :]
    counts = eligible.reshape(len(eligible), -1).sum(axis=1)
    if np.any(counts < 2):
        raise ValueError("Each sample needs at least two valid eligible tokens")
    n_mask = max(1, int(np.floor(counts.min() * mask_fraction)))
    n_mask = min(n_mask, int(counts.min()) - 1)
    mask = np.zeros_like(eligible)
    for batch_index in range(len(mask)):
        candidates = np.flatnonzero(eligible[batch_index].reshape(-1))
        chosen = rng.choice(candidates, size=n_mask, replace=False)
        mask[batch_index].reshape(-1)[chosen] = True
    forbidden = sorted(set(range(33)).difference(MASK_KEYPOINTS))
    assert not mask[:, :, forbidden].any()
    assert mask.reshape(len(mask), -1).any(axis=1).all()
    assert (~mask).reshape(len(mask), -1).any(axis=1).all()
    return mask


def mask_audit(mask, valid_patch):
    mask = np.asarray(mask, dtype=bool)
    valid_patch = np.asarray(valid_patch, dtype=bool)
    eligible_joint = np.zeros(33, dtype=bool)
    eligible_joint[MASK_KEYPOINTS] = True
    eligible = valid_patch & eligible_joint[None, None, :]
    masked_counts = mask.reshape(len(mask), -1).sum(axis=1)
    eligible_counts = eligible.reshape(len(mask), -1).sum(axis=1)
    per_sample_ratio = masked_counts / eligible_counts
    touched = np.flatnonzero(mask.any(axis=(0, 1))).tolist()
    return {
        "masked_keypoints": touched,
        "masked_names": [BLAZEPOSE_33[i] for i in touched],
        "global_fraction": float(mask.mean()),
        "eligible_mask_fraction_min": float(per_sample_ratio.min()),
        "eligible_mask_fraction_mean": float(per_sample_ratio.mean()),
        "eligible_mask_fraction_max": float(per_sample_ratio.max()),
        "forbidden_count": int(mask[:, :, sorted(set(range(33)) - set(MASK_KEYPOINTS))].sum()),
    }




In [ ]:
import copy
import math
import torch
from torch import nn


class SkeletonPatchEncoder(nn.Module):
    def __init__(
        self,
        frames=64,
        joints=33,
        coordinate_dim=3,
        segment_length=4,
        embed_dim=64,
        depth=2,
        heads=4,
        dropout=0.0,
    ):
        super().__init__()
        if frames % segment_length:
            raise ValueError("frames must be divisible by segment_length")
        self.frames = frames
        self.joints = joints
        self.coordinate_dim = coordinate_dim
        self.segment_length = segment_length
        self.segments = frames // segment_length
        self.embed_dim = embed_dim
        self.patch_embed = nn.Linear(segment_length * coordinate_dim, embed_dim)
        self.time_pos = nn.Parameter(torch.randn(self.segments, embed_dim) * 0.02)
        self.joint_pos = nn.Parameter(torch.randn(joints, embed_dim) * 0.02)
        layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=heads,
            dim_feedforward=embed_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.blocks = nn.TransformerEncoder(layer, num_layers=depth)
        self.norm = nn.LayerNorm(embed_dim)

    def patchify(self, x):
        batch, frames, joints, channels = x.shape
        expected = (self.frames, self.joints, self.coordinate_dim)
        if (frames, joints, channels) != expected:
            raise ValueError(f"Expected [B, {expected}], received {x.shape}")
        patches = x.reshape(
            batch, self.segments, self.segment_length, joints, channels
        )
        patches = patches.permute(0, 1, 3, 2, 4).contiguous()
        return patches.flatten(3)

    def positioned_tokens(self, x):
        tokens = self.patch_embed(self.patchify(x))
        return (
            tokens
            + self.time_pos[None, :, None, :]
            + self.joint_pos[None, None, :, :]
        )

    def forward(self, x, keep_mask=None):
        tokens = self.positioned_tokens(x)
        batch = len(tokens)
        flat = tokens.reshape(batch, self.segments * self.joints, self.embed_dim)
        if keep_mask is not None:
            keep_mask = keep_mask.reshape(batch, -1)
            kept_per_sample = keep_mask.sum(dim=1)
            if not torch.equal(kept_per_sample, kept_per_sample[:1].expand_as(kept_per_sample)):
                raise ValueError("Each sample must keep the same number of tokens")
            flat = flat[keep_mask].reshape(batch, int(kept_per_sample[0]), self.embed_dim)
        return self.norm(self.blocks(flat))


class SkeletonPredictor(nn.Module):
    def __init__(
        self,
        segments,
        joints,
        encoder_dim=64,
        predictor_dim=64,
        depth=2,
        heads=4,
        dropout=0.0,
    ):
        super().__init__()
        self.segments = segments
        self.joints = joints
        self.encoder_to_predictor = nn.Linear(encoder_dim, predictor_dim)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, predictor_dim))
        nn.init.normal_(self.mask_token, std=0.02)
        self.time_pos = nn.Parameter(torch.randn(segments, predictor_dim) * 0.02)
        self.joint_pos = nn.Parameter(torch.randn(joints, predictor_dim) * 0.02)
        layer = nn.TransformerEncoderLayer(
            d_model=predictor_dim,
            nhead=heads,
            dim_feedforward=predictor_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.blocks = nn.TransformerEncoder(layer, num_layers=depth)
        self.norm = nn.LayerNorm(predictor_dim)
        self.output = nn.Linear(predictor_dim, encoder_dim)

    def forward(self, visible_features, target_mask):
        batch = len(visible_features)
        target_mask = target_mask.reshape(batch, self.segments * self.joints)
        visible_mask = ~target_mask
        visible = self.encoder_to_predictor(visible_features)
        full = self.mask_token.expand(
            batch, self.segments * self.joints, -1
        ).clone()
        full[visible_mask] = visible.reshape(-1, visible.shape[-1])
        positions = (
            self.time_pos[:, None, :] + self.joint_pos[None, :, :]
        ).reshape(1, self.segments * self.joints, -1)
        full = full + positions
        predicted = self.output(self.norm(self.blocks(full)))
        return predicted[target_mask].reshape(batch, -1, predicted.shape[-1])


class SJEPAGait(nn.Module):
    def __init__(
        self,
        frames=64,
        joints=33,
        coordinate_dim=3,
        segment_length=4,
        embed_dim=64,
        encoder_depth=2,
        predictor_depth=2,
        heads=4,
    ):
        super().__init__()
        self.view_encoder = SkeletonPatchEncoder(
            frames, joints, coordinate_dim, segment_length,
            embed_dim, encoder_depth, heads,
        )
        self.target_encoder = copy.deepcopy(self.view_encoder)
        for parameter in self.target_encoder.parameters():
            parameter.requires_grad_(False)
        self.predictor = SkeletonPredictor(
            self.view_encoder.segments,
            joints,
            embed_dim,
            embed_dim,
            predictor_depth,
            heads,
        )
        self.register_buffer("target_center", torch.zeros(embed_dim))

    def forward(self, view, target, target_mask):
        visible_features = self.view_encoder(view, keep_mask=~target_mask)
        predicted = self.predictor(visible_features, target_mask)
        with torch.no_grad():
            all_targets = self.target_encoder(target)
            flat_mask = target_mask.reshape(len(target), -1)
            selected = all_targets[flat_mask].reshape(
                len(target), -1, all_targets.shape[-1]
            )
        return predicted, selected

    @torch.no_grad()
    def update_target(self, momentum):
        for target_parameter, view_parameter in zip(
            self.target_encoder.parameters(), self.view_encoder.parameters()
        ):
            target_parameter.mul_(momentum).add_(
                view_parameter, alpha=1.0 - momentum
            )

    @torch.no_grad()
    def update_center(self, targets, beta=0.9):
        batch_center = targets.mean(dim=(0, 1))
        self.target_center.mul_(beta).add_(batch_center, alpha=1.0 - beta)


def sjepa_cross_entropy(
    predicted,
    targets,
    center,
    predictor_temperature=0.10,
    target_temperature=0.06,
):
    target_prob = torch.softmax(
        (targets - center[None, None, :]) / target_temperature,
        dim=-1,
    ).detach()
    prediction_log_prob = torch.log_softmax(
        predicted / predictor_temperature,
        dim=-1,
    )
    return -(target_prob * prediction_log_prob).sum(dim=-1).mean()


def cosine_ema(step, total_steps, start=0.996, end=1.0):
    progress = min(max(step / max(total_steps - 1, 1), 0.0), 1.0)
    return end - (end - start) * (math.cos(math.pi * progress) + 1.0) / 2.0


LEFT_RIGHT_PAIRS = [
    (1, 4), (2, 5), (3, 6), (7, 8), (9, 10), (11, 12),
    (13, 14), (15, 16), (17, 18), (19, 20), (21, 22),
    (23, 24), (25, 26), (27, 28), (29, 30), (31, 32),
]


def geometric_view(
    x,
    max_degrees=8.0,
    translate=0.03,
    flip_probability=0.0,
):
    """Apply one sequence-wide transform per sample.

    Rotation is around the relative vertical y axis, so x and z are mixed.
    Flip defaults to off because laterality can matter for stroke. If enabled,
    coordinates are reflected and every left-right landmark pair is swapped.
    """
    view = x.clone()
    present = view.abs().sum(dim=-1) > 1e-8
    batch = len(view)
    angles = (
        torch.rand(batch, device=x.device) * 2.0 - 1.0
    ) * math.radians(max_degrees)
    cosine, sine = torch.cos(angles), torch.sin(angles)
    original_x = view[..., 0].clone()
    original_z = view[..., 2].clone()
    rotated_x = cosine[:, None, None] * original_x + sine[:, None, None] * original_z
    rotated_z = -sine[:, None, None] * original_x + cosine[:, None, None] * original_z
    view[..., 0] = rotated_x
    view[..., 2] = rotated_z
    offsets = (torch.rand(batch, 1, 1, 2, device=x.device) * 2.0 - 1.0) * translate
    view[..., :2] += offsets
    if flip_probability > 0:
        flip = torch.rand(batch, device=x.device) < flip_probability
        for batch_index in torch.where(flip)[0].tolist():
            view[batch_index, ..., 0] *= -1.0
            original = view[batch_index].clone()
            original_present = present[batch_index].clone()
            for left, right in LEFT_RIGHT_PAIRS:
                view[batch_index, :, left] = original[:, right]
                view[batch_index, :, right] = original[:, left]
                present[batch_index, :, left] = original_present[:, right]
                present[batch_index, :, right] = original_present[:, left]
    view = view.masked_fill(~present[..., None], 0.0)
    return view




In [ ]:
def pose_records_from_cache(pose_dir=POSE_DIR, conditions=CONDITIONS):
    records = []
    for condition in conditions:
        folder = Path(pose_dir) / condition
        for path in sorted(folder.glob("*.npz")):
            data = np.load(path, allow_pickle=False)
            required = {
                "sequence", "sequence_id", "video_id", "condition",
                "frame_numbers", "crop_bounds", "fps", "source_csv",
                "source_video", "pose_model", "pose_model_sha256",
                "extraction_version",
            }
            missing = required.difference(data.files)
            if missing:
                raise ValueError(
                    f"Stale pose cache {path} is missing {sorted(missing)}. "
                    "Re-extract it with notebook 02."
                )
            sequence = data["sequence"].astype(np.float32)
            if sequence.ndim != 3 or sequence.shape[1:] != (33, 4):
                raise ValueError(f"Bad pose shape in {path}: {sequence.shape}")
            stored_condition = str(data["condition"].item())
            if stored_condition != condition:
                raise ValueError(
                    f"Pose condition {stored_condition} does not match folder {condition}"
                )
            if len(data["frame_numbers"]) != len(sequence):
                raise ValueError(f"Frame and pose lengths differ in {path}")
            records.append({
                "condition": condition,
                "sequence_id": str(data["sequence_id"].item()),
                "video_id": str(data["video_id"].item()),
                "source_video": str(data["source_video"].item()),
                "fps": float(data["fps"].item()),
                "extraction_version": str(data["extraction_version"].item()),
                "pose_model_sha256": str(data["pose_model_sha256"].item()),
                "sequence": sequence,
                "path": str(path),
            })
    return records


def load_records_for_mode(conditions=CONDITIONS, smoke_per_condition=10, frames=64):
    if MODE == "smoke":
        records = synthetic_corpus(
            conditions=conditions,
            n_per_condition=smoke_per_condition,
            frames=frames,
        )
        print(f"Explicit smoke corpus: {len(records)} synthetic sequences")
        return records
    records = pose_records_from_cache(conditions=conditions)
    counts = pd.Series([r["condition"] for r in records]).value_counts()
    missing = [condition for condition in conditions if counts.get(condition, 0) == 0]
    if missing:
        raise FileNotFoundError(
            f"Real mode requires cached pose sequences for {missing}. "
            "Run notebook 02 first."
        )
    print(f"Real pose corpus: {len(records)} sequences")
    return records




## Load the twelve normal sequences and build the future mask

Only the twelve normal sequences are used for training, exactly as in notebook 04. Preprocessing is unchanged: short internal gaps are interpolated, clips are centered on the mid-hip and scaled by body width, then temporally resized to FRAMES and reduced to x, y, relative z. Ends and long gaps stay invalid and can never become loss targets.

### The causal mask geometry

Temporal segment s covers frames 4s to 4s+3. The future mask hides the last 4 of the 16 segments for every one of the 33 joints. The last visible frame is frame 47, so the hidden future spans frames 48 to 63, sixteen frames of motion.

| Quantity | Value |
|---|---|
| Frames per clip | 64 |
| Segment length | 4 frames |
| Temporal segments | 16 |
| Context segments (view encoder) | 12, frames 0 to 47 |
| Future segments (targets) | 4, frames 48 to 63 |
| Joints inside the future mask | all 33 |
| Visible tokens per sample | 12 x 33 = 396 |
| Future target tokens per sample | 4 x 33 = 132 |
| Teacher tokens per sample | 16 x 33 = 528 |
| Horizon of future segment h | 4h frames past frame 47 |

The values assume the default 64-frame layout; the code derives every number from FRAMES, so an override with fewer frames still runs. Three implementation points keep the mask honest.

1. The view encoder never sees the future. SJEPAGait passes ~target_mask to SkeletonPatchEncoder as keep_mask, so the encoder's transformer receives only the 396 context tokens. The shared class requires every sample in a batch to keep the same number of tokens, because a transformer layer needs a rectangular input. The future mask is the same rectangle for all samples, so the invariant holds by construction. The configuration cell verifies it three ways: a batch that keeps 240 tokens per sample, the causal 396-token keep mask, and an unequal keep mask that must raise the documented ValueError.

2. The loss supervises only future positions. The predictor still fills all 528 slots, but the shared step gathers predictions and teacher latents only at the 132 masked slots, so contemporaneous positions contribute nothing to the gradient.

3. Validity still matters. The future window can contain frames where the pose detector failed. Those tokens keep the mask geometry, so batches stay rectangular, but they receive a zero weight in the loss, exactly as notebook 04 never lets invalid patches become targets.

### Why the teacher still sees the whole clip

The teacher encoder receives the complete clip, future included. This mirrors notebook 04 and it is a deliberate choice. The EMA teacher must be a stable, slowly moving target; if it also received only the past, the learning signal would have to be bootstrapped from the predictor's own guesses, which is very hard to stabilize on a twelve-clip dataset. Giving the teacher the future means each target latent truly describes what that future segment is, so the loss measures a genuine forecast error against an oracle summary.

The limitation that comes with the choice is real. At deployment there is no future clip for the teacher. A real gait world model would have to roll the predictor forward: predict the next latent, treat it as context, predict again, and pay a compounding-error cost that training never shows. Read this notebook as "can the encoder and predictor learn a causal, oracle-supervised forecast?", not as "is the closed loop stable?". An online, teacher-free version with self-consistency losses is future work, and the discussion section returns to it.

In [ ]:
FRAMES = int(os.getenv("SJEPA_FRAMES", "64"))
SEGMENT_LENGTH = 4
if FRAMES % SEGMENT_LENGTH:
    raise ValueError("SJEPA_FRAMES must be divisible by 4")
SEGMENTS = FRAMES // SEGMENT_LENGTH
FUTURE_SEGMENTS = int(os.getenv("SJEPA_FUTURE_SEGMENTS", "4"))
if FUTURE_SEGMENTS >= SEGMENTS:
    raise ValueError("SJEPA_FUTURE_SEGMENTS must leave at least one context segment")
KEEP_SEGMENTS = SEGMENTS - FUTURE_SEGMENTS

normal_records = load_records_for_mode(
    conditions=["normal"],
    smoke_per_condition=12,
    frames=FRAMES,
)
assert {record["condition"] for record in normal_records} == {"normal"}
expected_normal = int(os.getenv("GAVD_EXPECTED_NORMAL_SEQUENCES", "12"))
if MODE == "real" and expected_normal and len(normal_records) != expected_normal:
    raise ValueError(
        f"Expected {expected_normal} normal pose files, found {len(normal_records)}. "
        "In penny/gavd3/.env set GAVD_EXTRACT_POSES=1, "
        "GAVD_EXTRACT_CONDITIONS=normal, and GAVD_MAX_SEQUENCES=0; "
        "restart the kernel and rerun notebook 02 through its extraction cell."
    )
prepared = [
    prepare_sequence(record["sequence"], frames=FRAMES)
    for record in normal_records
]
all_xyz = np.stack([item[0] for item in prepared])
all_valid = np.stack([item[1] for item in prepared])
sequence_ids = [record["sequence_id"] for record in normal_records]
video_ids = [record["video_id"] for record in normal_records]
min_coverage = float(os.getenv("GAVD_MIN_NEURO_COVERAGE", "0.50"))
coverage_report = pd.DataFrame({
    "sequence_id": sequence_ids,
    "video_id": video_ids,
    "neurologic_observed_fraction": all_valid[:, :, MASK_KEYPOINTS].mean(axis=(1, 2)),
})
display(coverage_report)
coverage_report.to_csv(ARTIFACT_DIR / "10_normal_pose_coverage.csv", index=False)
below = coverage_report[
    coverage_report["neurologic_observed_fraction"] < min_coverage
]
if not below.empty:
    raise ValueError(
        f"{len(below)} normal sequences fall below the neurologic "
        f"coverage threshold {min_coverage:.2f}. Review extraction first."
    )


def causal_future_mask(count, segments, future_segments, joints=33):
    '''True marks a future target token: hidden from the view encoder and
    supervised through the latent objective.'''
    mask = np.zeros((count, segments, joints), dtype=bool)
    mask[:, segments - future_segments:, :] = True
    return mask


future_mask_np = causal_future_mask(1, SEGMENTS, FUTURE_SEGMENTS)[0]
visible_per_sample = int((~future_mask_np).sum())
target_per_sample = int(future_mask_np.sum())
assert not future_mask_np[:KEEP_SEGMENTS, :].any()
assert future_mask_np[KEEP_SEGMENTS:, :].all()
print("normal tensor:", all_xyz.shape)
print(
    "temporal segments:", SEGMENTS,
    "context segments:", KEEP_SEGMENTS,
    "future segments:", FUTURE_SEGMENTS,
)
print(
    "visible tokens per sample:", visible_per_sample,
    "= context segments x 33 joints",
)
print(
    "future target tokens per sample:", target_per_sample,
    "= future segments x 33 joints",
)

audit = mask_audit(
    np.broadcast_to(future_mask_np[None], (2, SEGMENTS, 33)),
    np.ones((2, SEGMENTS, 33), dtype=bool),
)
audit_frame = pd.DataFrame([audit])
display(audit_frame)
audit_frame.to_csv(ARTIFACT_DIR / "10_future_mask_audit.csv", index=False)
print("future mask audited: whole-body by design (masked_names lists all 33 joints)")
print("normal source videos:", len(set(video_ids)))
if MODE == "real" and len(set(video_ids)) == 1:
    print(
        "Warning: all normal sequences share one source video. "
        "Forecasting here is transductive with respect to that video."
    )
if MODE == "smoke":
    print("SMOKE fixture outputs are code-path checks only, not clinical results.")

## Configure a compact causal model

The causal variant keeps the identical SJEPAGait architecture. Only the mask changes, which is the point of reusing the shared code. Sizes follow the tutorial-scale philosophy of notebook 04: smoke stays tiny so the full graph runs on a laptop, while real mode has a `quick` profile that validates the pipeline and a `recommended` profile for the substantive 300-epoch run.

| Setting | Smoke | Real quick | Real recommended |
|---|---:|---:|---:|
| Embedding width | 32 | 96 | 96 |
| Encoder depth | 1 | 4 | 4 |
| Predictor depth | 1 | 2 | 2 |
| Epochs | 2 | 20 | 300 |
| Optimizer updates (12 clips, batch 4) | 6 | 60 | 900 |
| Env switch | none | `SJEPA_RUN_PROFILE=quick` | default |

The cell below also verifies the equal-kept-count contract of `SkeletonPatchEncoder` before training starts. A Transformer needs a rectangular input, so every sample in a batch must keep the same number of tokens. The causal mask is a fixed rectangle, which satisfies the contract automatically; the verification feeds a batch that keeps 240 tokens per sample, a batch that keeps the causal 396 tokens per sample, and a deliberately unequal mask that must raise the documented ValueError.

In [ ]:
if MODE == "smoke":
    RUN_PROFILE = "smoke"
    EMBED_DIM, ENCODER_DEPTH, PREDICTOR_DEPTH, HEADS = 32, 1, 1, 4
    EPOCHS = int(os.getenv("SJEPA_EPOCHS", "2"))
    BATCH_SIZE = 4
    EMA_START = 0.996
else:
    RUN_PROFILE = os.getenv(
        "SJEPA_RUN_PROFILE", "recommended"
    ).strip().lower()
    if RUN_PROFILE not in {"quick", "recommended"}:
        raise ValueError(
            "SJEPA_RUN_PROFILE must be quick or recommended"
        )
    EMBED_DIM, ENCODER_DEPTH, PREDICTOR_DEPTH, HEADS = 96, 4, 2, 4
    default_epochs = "20" if RUN_PROFILE == "quick" else "300"
    default_ema = "0.996" if RUN_PROFILE == "quick" else "0.999"
    EPOCHS = int(os.getenv("SJEPA_EPOCHS", default_epochs))
    BATCH_SIZE = int(os.getenv("SJEPA_BATCH_SIZE", "4"))
    EMA_START = float(os.getenv("SJEPA_EMA_START", default_ema))

config = {
    "frames": FRAMES,
    "joints": 33,
    "coordinate_dim": 3,
    "segment_length": SEGMENT_LENGTH,
    "embed_dim": EMBED_DIM,
    "encoder_depth": ENCODER_DEPTH,
    "predictor_depth": PREDICTOR_DEPTH,
    "heads": HEADS,
}
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
torch.manual_seed(42)
np.random.seed(42)
model = SJEPAGait(**config).to(device)
trainable = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(
    trainable,
    lr=3e-4 if MODE == "smoke" else 1e-3,
    betas=(0.9, 0.95),
    weight_decay=0.05,
)
target_mask_all = torch.tensor(future_mask_np, dtype=torch.bool, device=device)
print("device:", device)
print("training profile:", RUN_PROFILE)
if MODE == "real" and RUN_PROFILE == "quick":
    print("QUICK PROFILE: validate the pipeline only; do not report its score.")
print("trainable parameters:", sum(p.numel() for p in trainable))
print(
    "target trainable parameters:",
    sum(p.numel() for p in model.target_encoder.parameters() if p.requires_grad),
)
assert not any(p.requires_grad for p in model.target_encoder.parameters())

with torch.no_grad():
    toy = torch.randn(2, FRAMES, 33, 3, device=device)
    equal_keep = torch.zeros(2, SEGMENTS * 33, dtype=torch.bool, device=device)
    equal_keep[:, :240] = True
    kept_240 = model.view_encoder(toy, keep_mask=equal_keep)
    assert kept_240.shape == (2, 240, EMBED_DIM)

    causal_keep = ~target_mask_all.reshape(1, -1).expand(3, -1)
    kept_causal = model.view_encoder(
        torch.randn(3, FRAMES, 33, 3, device=device),
        keep_mask=causal_keep,
    )
    assert kept_causal.shape == (3, visible_per_sample, EMBED_DIM)

    unequal = equal_keep.clone()
    unequal[1, 0] = False
    try:
        model.view_encoder(toy, keep_mask=unequal)
    except ValueError as error:
        print("unequal-keep guard message:", error)

print("equal-keep 240-token output shape:", tuple(kept_240.shape))
print("causal keep output shape:", tuple(kept_causal.shape))
print("visible tokens per sample:", visible_per_sample, "(context segments x 33 joints)")

## Train a causal forecaster on the twelve normal sequences

The training step is notebook 04 with three substitutions. The target mask is the fixed future rectangle instead of a random neurologic draw. The loss is the shared latent cross-entropy, but computed only on future tokens and weighted by future validity, since an invalid future patch must not become a target. `geometric_view` still perturbs only the view input. Everything else, cosine EMA from the view encoder to the teacher, target centering, sharpened target probabilities, gradient clipping, and the per-epoch collapse diagnostics, is untouched.

Two smoke epochs exercise the code path and nothing else; the numbers are random-guess numbers. A healthy real run should show the future cross-entropy falling over hundreds of epochs while the feature standard deviation stays well above zero and the mean pairwise cosine stays below one, the same collapse warnings notebook 04 used.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.functional as F


def latent_cross_entropy_per_token(
    predicted,
    targets,
    center,
    predictor_temperature=0.10,
    target_temperature=0.06,
):
    target_prob = torch.softmax(
        (targets - center[None, None, :]) / target_temperature,
        dim=-1,
    ).detach()
    prediction_log_prob = torch.log_softmax(
        predicted / predictor_temperature,
        dim=-1,
    )
    return -(target_prob * prediction_log_prob).sum(dim=-1)


def weighted_future_loss(predicted, targets, center, weights):
    per_token = latent_cross_entropy_per_token(predicted, targets, center)
    denominator = weights.sum().clamp_min(1.0)
    return (per_token * weights).sum() / denominator


dataset = TensorDataset(
    torch.tensor(all_xyz, dtype=torch.float32),
    torch.tensor(all_valid, dtype=torch.bool),
)
generator = torch.Generator().manual_seed(42)
loader = DataLoader(
    dataset,
    batch_size=min(BATCH_SIZE, len(dataset)),
    shuffle=True,
    generator=generator,
    drop_last=False,
)
total_steps = EPOCHS * len(loader)
warmup_steps = max(1, min(len(loader), total_steps // 10))
minimum_initial_teacher_retention = EMA_START ** total_steps
print("optimizer updates:", total_steps)
print("minimum initial-teacher retention under the EMA schedule:", f"{minimum_initial_teacher_retention:.3f}")


def learning_rate_factor(step):
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(total_steps - warmup_steps - 1, 1)
    return 0.5 + 0.5 * (1.0 + math.cos(math.pi * progress)) / 2.0


scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lr_lambda=learning_rate_factor
)


@torch.no_grad()
def collapse_diagnostics(model, arrays, batch_size=8):
    model.target_encoder.eval()
    pooled = []
    for start in range(0, len(arrays), batch_size):
        batch = torch.tensor(
            arrays[start:start + batch_size],
            dtype=torch.float32,
            device=device,
        )
        tokens = model.target_encoder(batch)
        pooled.append(tokens.mean(dim=1).cpu())
    pooled = torch.cat(pooled)
    feature_std = pooled.std(dim=0, unbiased=False).mean().item()
    normalized = F.normalize(pooled, dim=1)
    similarities = normalized @ normalized.T
    if len(pooled) > 1:
        off_diagonal = similarities[~torch.eye(
            len(pooled), dtype=torch.bool
        )].mean().item()
    else:
        off_diagonal = float("nan")
    return feature_std, off_diagonal


history = []
global_step = 0
target_before = next(model.target_encoder.parameters()).detach().clone()
for epoch in range(EPOCHS):
    model.train()
    batch_rows = []
    for coordinates, valid in loader:
        coordinates = coordinates.to(device)
        valid = valid.to(device)
        valid_patch = valid.reshape(
            len(valid), SEGMENTS, SEGMENT_LENGTH, 33
        ).all(dim=2)
        target_mask = target_mask_all[None].expand(len(valid), -1, -1)
        weights = valid_patch[:, KEEP_SEGMENTS:, :].reshape(len(valid), -1).float()
        view = geometric_view(
            coordinates,
            max_degrees=8.0,
            translate=0.03,
            flip_probability=0.0,
        )
        prediction, target = model(view, coordinates, target_mask)
        loss = weighted_future_loss(
            prediction, target, model.target_center, weights
        )
        if not torch.isfinite(loss):
            raise FloatingPointError(f"Non-finite loss at step {global_step}")
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        assert all(
            parameter.grad is None
            for parameter in model.target_encoder.parameters()
        )
        torch.nn.utils.clip_grad_norm_(trainable, max_norm=1.0)
        optimizer.step()
        scheduler.step()
        momentum = cosine_ema(
            global_step, total_steps, start=EMA_START, end=1.0
        )
        model.update_target(momentum)
        model.update_center(target, beta=0.9)

        with torch.no_grad():
            keep_rows = weights > 0.5
            if keep_rows.any():
                centered = target[keep_rows] - model.target_center[None, :]
                target_prob = torch.softmax(centered / 0.06, dim=-1)
                predicted_prob = torch.softmax(
                    prediction[keep_rows] / 0.10, dim=-1
                )
                target_entropy = (
                    -(target_prob * target_prob.clamp_min(1e-8).log())
                    .sum(dim=-1)
                    .mean()
                    .item()
                )
                predictor_entropy = (
                    -(predicted_prob * predicted_prob.clamp_min(1e-8).log())
                    .sum(dim=-1)
                    .mean()
                    .item()
                )
            else:
                target_entropy = float("nan")
                predictor_entropy = float("nan")
        batch_rows.append({
            "loss": float(loss.detach().cpu()),
            "valid_future_fraction": float(weights.mean().cpu()),
            "target_entropy": target_entropy,
            "predictor_entropy": predictor_entropy,
            "ema_momentum": momentum,
        })
        global_step += 1

    feature_std, mean_cosine = collapse_diagnostics(model, all_xyz)
    summary = pd.DataFrame(batch_rows).mean(numeric_only=True).to_dict()
    summary.update({
        "epoch": epoch + 1,
        "feature_std": feature_std,
        "mean_pair_cosine": mean_cosine,
        "learning_rate": optimizer.param_groups[0]["lr"],
    })
    history.append(summary)
    print(
        f"epoch {epoch + 1:03d}  "
        f"loss {summary['loss']:.4f}  "
        f"std {feature_std:.4f}  "
        f"cos {mean_cosine:.4f}"
    )

history_df = pd.DataFrame(history)
target_after = next(model.target_encoder.parameters()).detach()
ema_change = float((target_after - target_before).abs().mean().cpu())
assert ema_change > 0.0
assert np.isfinite(history_df.select_dtypes("number")).all().all()
print("mean absolute EMA target change:", ema_change)
display(history_df)

model = model.cpu()
device = torch.device("cpu")
target_mask_all = target_mask_all.cpu()
print("model moved to CPU for downstream evaluation")

torch.save(
    {
        "model_state": model.state_dict(),
        "config": config,
        "mode": MODE,
        "objective": "causal_future_prediction",
        "future_segments": FUTURE_SEGMENTS,
        "keep_segments": KEEP_SEGMENTS,
        "sequence_ids": sequence_ids,
        "video_ids": video_ids,
        "seed": 42,
    },
    ARTIFACT_DIR / "sjepa_causal.pt",
)
print("saved causal checkpoint:", ARTIFACT_DIR / "sjepa_causal.pt")

## Save the training history

The per-epoch history is written to the artifact folder under this mode, with the same naming pattern as every other notebook in the family. A smoke run writes to the smoke artifact folder, so it can never overwrite a real run's file.

In [ ]:
history_df.to_csv(ARTIFACT_DIR / "10_training_history.csv", index=False)
print("saved:", ARTIFACT_DIR / "10_training_history.csv")
print("history rows:", len(history_df))
print("final future cross-entropy:", f"{history_df['loss'].iloc[-1]:.4f}")

## Shared family scaffolding, reproduced verbatim

The S-JEPA notebooks in this folder pool each clip into one fixed-length vector with the masked mean and standard deviation helper, over all tokens and over the neurologic subset. This notebook works at token level and never classifies, but the helper is family scaffolding and convention 9 of the shared spec requires it to be copied rather than imported. It runs once on the frozen causal teacher below to confirm the causal model exposes the same interface.

In [ ]:
def masked_mean_std(tokens, mask):
    weights = torch.as_tensor(
        mask, dtype=tokens.dtype, device=tokens.device
    ).unsqueeze(-1)
    denominator = weights.sum(dim=1).clamp_min(1.0)
    mean = (tokens * weights).sum(dim=1) / denominator
    variance = (
        (tokens - mean[:, None, :]).square() * weights
    ).sum(dim=1) / denominator
    return mean, variance.clamp_min(0.0).sqrt()


@torch.no_grad()
def pooled_embeddings(model, arrays, validity, batch_size=8):
    vectors = []
    segments = model.target_encoder.segments
    segment_length = model.target_encoder.segment_length
    dimension = model.target_encoder.embed_dim
    for start in range(0, len(arrays), batch_size):
        batch = torch.tensor(
            arrays[start:start + batch_size],
            dtype=torch.float32,
        )
        tokens = model.target_encoder(batch).reshape(
            len(batch), segments, 33, dimension
        )
        valid_patch = np.asarray(
            validity[start:start + batch_size], dtype=bool
        ).reshape(
            len(batch), segments, segment_length, 33
        ).all(axis=2)
        global_tokens = tokens.reshape(len(batch), -1, dimension)
        neuro_tokens = tokens[:, :, MASK_KEYPOINTS].reshape(
            len(batch), -1, dimension
        )
        global_mean, global_std = masked_mean_std(
            global_tokens, valid_patch.reshape(len(batch), -1)
        )
        neuro_mean, neuro_std = masked_mean_std(
            neuro_tokens,
            valid_patch[:, :, MASK_KEYPOINTS].reshape(len(batch), -1),
        )
        vector = torch.cat(
            [
                global_mean,
                global_std,
                neuro_mean,
                neuro_std,
            ],
            dim=1,
        )
        vectors.append(vector.cpu())
    return torch.cat(vectors).numpy()






## Evaluate the forecast horizon by horizon

The four future segments sit 4, 8, 12, and 16 frames past the last visible frame, so the table reports four horizons. For each horizon the model's latent cross-entropy is computed on the future tokens of the twelve normal clips, weighted by future validity. The loss is evaluated on the same sharp teacher targets that training used, so the metric has the same meaning as the training curve.

The baseline is a phase-bin mean-latent predictor. For each held-out clip it estimates the continuous gait phase from the last observed context segment, using the Hilbert transform of the centered left-ankle x signal, and buckets that phase into six bins. Then, for each future segment, it averages the teacher latents of that segment over the eleven remaining clips whose context phase falls in the same bin, pooled over the 33 joint tokens into one mean latent per horizon, and broadcasts that mean back to the segment's tokens. If the held-out bin has no training member, the baseline falls back to all eleven clips. This is a true leave-one-out procedure over the twelve clips: a clip's own future never leaks into its baseline prediction.

The baseline is deliberately crude. It answers "what does an average future latent look like for this phase of the stride?", it discards every joint and person-specific detail, and it receives the phase as free information that the model had to learn to encode. If the causal model does not clearly beat it, the representation is not carrying much forecastable dynamics.

Both curves are reported in latent cross-entropy nats. Because the target distribution is sharpened before the softmax, raw values are large, so the figure plots this panel on a log axis.

In [ ]:
from scipy.signal import hilbert

model.eval()
center = model.target_center


def ankle_x_phase(xyz):
    '''Full-clip Hilbert phase of the left-ankle x signal. Used ONLY as the
    ground-truth target for future frames; it sees the future by design.'''
    signal = xyz[:, 27, 0].astype(np.float64)
    signal = signal - signal.mean()
    analytic = hilbert(signal)
    return np.mod(np.angle(analytic), 2.0 * np.pi)


def causal_context_phase(xyz, keep_frames):
    '''Causal phase: Hilbert transform over the CONTEXT prefix only, so the
    baseline and persist references never see the frames being predicted.'''
    signal = xyz[:keep_frames, 27, 0].astype(np.float64)
    signal = signal - signal.mean()
    analytic = hilbert(signal)
    return np.mod(np.angle(analytic), 2.0 * np.pi)


KEEP_FRAMES = SEGMENT_LENGTH * KEEP_SEGMENTS
phase_frame = np.stack([ankle_x_phase(seq) for seq in all_xyz])
phase_context = np.stack([
    causal_context_phase(seq, KEEP_FRAMES)[-1] for seq in all_xyz
])
PHASE_BINS = 6
bin_index = np.floor(
    phase_context / (2.0 * np.pi) * PHASE_BINS
).astype(int) % PHASE_BINS


@torch.no_grad()
def forecast_errors(model, xyz, valid, center):
    xyz_tensor = torch.tensor(np.asarray(xyz, dtype=np.float32))
    valid_tensor = torch.tensor(np.asarray(valid, dtype=bool))
    target_mask = target_mask_all[None].expand(len(xyz_tensor), -1, -1)
    prediction, teacher = model(xyz_tensor, xyz_tensor, target_mask)
    per_token = latent_cross_entropy_per_token(prediction, teacher, center)
    count = len(xyz_tensor)
    valid_patch = valid_tensor.reshape(
        count, SEGMENTS, SEGMENT_LENGTH, 33
    ).all(dim=2)
    weights = valid_patch[:, KEEP_SEGMENTS:, :].reshape(count, -1).float()
    per_seq = (per_token * weights).sum(dim=1) / weights.sum(dim=1).clamp_min(1.0)
    prediction = prediction.reshape(count, FUTURE_SEGMENTS, 33, EMBED_DIM)
    teacher = teacher.reshape(count, FUTURE_SEGMENTS, 33, EMBED_DIM)
    per_h = per_token.reshape(count, FUTURE_SEGMENTS, 33)
    weights_h = weights.reshape(count, FUTURE_SEGMENTS, 33)
    return prediction, teacher, per_h, weights_h, per_seq


normal_prediction, normal_teacher, normal_per_h, normal_w_h, normal_per_seq = (
    forecast_errors(model, all_xyz, all_valid, center)
)
NORMAL_COUNT = len(all_xyz)

teacher_pooled = normal_teacher.mean(dim=2)  # pool the 33 joint tokens per horizon
baseline_flat = torch.empty(NORMAL_COUNT, FUTURE_SEGMENTS, 33, EMBED_DIM)
for sample_index in range(NORMAL_COUNT):
    others = [j for j in range(NORMAL_COUNT) if j != sample_index]
    members = [j for j in others if bin_index[j] == bin_index[sample_index]]
    if not members:
        members = list(others)
    for horizon in range(FUTURE_SEGMENTS):
        template = teacher_pooled[members, horizon].mean(dim=0)
        baseline_flat[sample_index, horizon] = template[None, :].expand(33, -1)

baseline_per_h = latent_cross_entropy_per_token(
    baseline_flat.reshape(NORMAL_COUNT, -1, EMBED_DIM),
    normal_teacher.reshape(NORMAL_COUNT, -1, EMBED_DIM),
    center,
).reshape(NORMAL_COUNT, FUTURE_SEGMENTS, 33)

horizon_rows = []
for horizon in range(FUTURE_SEGMENTS):
    denominator = normal_w_h[:, horizon, :].sum().clamp_min(1.0)
    horizon_rows.append({
        "horizon": horizon + 1,
        "horizon_frames": (horizon + 1) * SEGMENT_LENGTH,
        "sequences": NORMAL_COUNT,
        "model_future_ce": float(
            (normal_per_h[:, horizon, :] * normal_w_h[:, horizon, :]).sum()
            / denominator
        ),
        "baseline_future_ce": float(
            (baseline_per_h[:, horizon, :] * normal_w_h[:, horizon, :]).sum()
            / denominator
        ),
    })
horizon_frame = pd.DataFrame(horizon_rows)
display(horizon_frame)
horizon_frame.to_csv(ARTIFACT_DIR / "10_horizon_prediction.csv", index=False)

normal_rows = [{
    "sequence_id": sequence_ids[sample_index],
    "video_id": video_ids[sample_index],
    "condition": "normal",
    "forecast_ce": float(normal_per_seq[sample_index].cpu()),
    "valid_future_fraction": float(
        normal_w_h[sample_index].sum().cpu() / (FUTURE_SEGMENTS * 33)
    ),
} for sample_index in range(NORMAL_COUNT)]
normal_errors = pd.DataFrame(normal_rows)
normal_errors.to_csv(ARTIFACT_DIR / "10_sequence_forecast_errors.csv", index=False)
print("saved:", ARTIFACT_DIR / "10_horizon_prediction.csv")
print("saved:", ARTIFACT_DIR / "10_sequence_forecast_errors.csv")
print(
    "phase-context bin counts (leave-one-out over the 12 clips):",
    pd.Series(bin_index).value_counts().sort_index().to_dict(),
)

## Out-of-distribution check on abnormal gait

The forecaster was trained on twelve normal clips only. Its OOD test uses the four abnormal conditions from the same pose cache, none of which appeared in training: parkinsons, stroke, cerebralpalsy, and myopathic. Prediction error is measured with the identical horizon loss on future tokens.

The expected result is that abnormal gait forecasts worse than normal gait. Pathological walking is not part of the dynamics the model learned, and every abnormal video is a different person and recording than the normal source video, so identity and camera also change. Both are part of what "out of distribution" means here, and both are confounds, exactly the caveat notebook 07 raised. Treat the direction as a claim to check with the printed numbers, not as a given. In smoke mode the synthetic conditions are code-path fixtures with no clinical meaning, and no trend should be asserted.

In [ ]:
abnormal_conditions = [condition for condition in CONDITIONS if condition != "normal"]
abnormal_records = load_records_for_mode(
    conditions=abnormal_conditions,
    smoke_per_condition=12,
    frames=FRAMES,
)
abn_prepared = [
    prepare_sequence(record["sequence"], frames=FRAMES)
    for record in abnormal_records
]
abn_xyz = np.stack([item[0] for item in abn_prepared])
abn_valid = np.stack([item[1] for item in abn_prepared])
abn_condition = [record["condition"] for record in abnormal_records]
abn_sequence_ids = [record["sequence_id"] for record in abnormal_records]
abn_video_ids = [record["video_id"] for record in abnormal_records]
model.eval()
abn_prediction, abn_teacher, abn_per_h, abn_w_h, abn_per_seq = forecast_errors(
    model, abn_xyz, abn_valid, center
)
print("abnormal tensor:", abn_xyz.shape)

all_rows = list(normal_rows)
for sample_index in range(len(abnormal_records)):
    all_rows.append({
        "sequence_id": abn_sequence_ids[sample_index],
        "video_id": abn_video_ids[sample_index],
        "condition": abn_condition[sample_index],
        "forecast_ce": float(abn_per_seq[sample_index].cpu()),
        "valid_future_fraction": float(
            abn_w_h[sample_index].sum().cpu() / (FUTURE_SEGMENTS * 33)
        ),
    })
sequence_errors_all = pd.DataFrame(all_rows)
sequence_errors_all.to_csv(
    ARTIFACT_DIR / "10_sequence_forecast_errors_all.csv", index=False
)

condition_summary = (
    sequence_errors_all
    .groupby("condition", sort=False)
    .agg(
        n_sequences=("forecast_ce", "size"),
        mean_forecast_ce=("forecast_ce", "mean"),
        mean_valid_future_fraction=("valid_future_fraction", "mean"),
    )
    .reset_index()
)
condition_order = ["normal"] + abnormal_conditions
condition_summary = condition_summary.set_index("condition").loc[condition_order].reset_index()
display(condition_summary)
condition_summary.to_csv(ARTIFACT_DIR / "10_ood_prediction_errors.csv", index=False)
print("saved:", ARTIFACT_DIR / "10_ood_prediction_errors.csv")
for _, row in condition_summary.iterrows():
    print(
        f"{row['condition']:<14s} n={int(row['n_sequences']):3d}  "
        f"mean forecast CE {row['mean_forecast_ce']:.4f}"
    )
if MODE == "real":
    print(
        "Real-mode expectation: abnormal conditions should sit above normal "
        "because their dynamics were never trained on. Read the numbers first."
    )
    print(
        "Caution: the normal row IS the training set (the 12 clips the "
        "forecaster was trained on), so it is a transductive reference, not "
        "an estimate of error on unseen normal gait."
    )
    low_coverage = sequence_errors_all[
        sequence_errors_all["valid_future_fraction"] < 0.5
    ]
    if len(low_coverage):
        print(
            "Flagged low-coverage future windows (valid_future_fraction < 0.5):"
        )
        display(low_coverage)
    else:
        print("No low-coverage future windows to flag.")

## Latent rollout probe: does the predicted future keep the phase clock?

Notebook 08 probes how well latents decode continuous gait phase. This probe asks a harder question: do the predicted future latents, which the model produced without ever seeing the future, still encode the phase that the future actually has?

For each normal clip and each future segment, the probe pools the segment's predicted tokens over joints into one vector, then trains a Ridge regressor from those vectors to the cosine and sine of the true instantaneous phase of that segment, measured with the Hilbert transform of the centered left-ankle x signal. Validation is leave-one-out over the twelve clips, so a clip's own latents are never used to fit its regressor. Two references are decoded with the identical procedure: the teacher latents of the same segments, which see the future and are an upper reference, and a persist baseline that simply repeats the context phase, a driftless clock that ignores the segment entirely.

In [ ]:
from sklearn.linear_model import Ridge


true_phase = np.stack([
    phase_frame[:, SEGMENT_LENGTH * (KEEP_SEGMENTS + horizon) + 2]
    for horizon in range(FUTURE_SEGMENTS)
], axis=1)
true_phase = np.mod(true_phase, 2.0 * np.pi)

model_pooled = normal_prediction.mean(dim=2).numpy()
teacher_pooled_np = normal_teacher.mean(dim=2).numpy()


def decode_phase_leave_one_out(features, true_angle):
    count, horizons, dim = features.shape
    targets = np.stack([np.cos(true_angle), np.sin(true_angle)], axis=-1)
    flat_x = features.reshape(-1, dim)
    flat_y = targets.reshape(-1, 2)
    decoded = np.zeros_like(true_angle)
    for sample_index in range(count):
        test_rows = np.arange(sample_index * horizons, (sample_index + 1) * horizons)
        train_rows = np.setdiff1d(np.arange(count * horizons), test_rows)
        ridge = Ridge(alpha=1.0).fit(flat_x[train_rows], flat_y[train_rows])
        predicted_cs = ridge.predict(flat_x[test_rows])
        decoded[sample_index] = np.mod(
            np.arctan2(predicted_cs[:, 1], predicted_cs[:, 0]),
            2.0 * np.pi,
        )
    return decoded


def circular_error(predicted, true_angle):
    delta = np.mod(predicted - true_angle + np.pi, 2.0 * np.pi) - np.pi
    return np.abs(delta)


model_angle = decode_phase_leave_one_out(model_pooled, true_phase)
teacher_angle = decode_phase_leave_one_out(teacher_pooled_np, true_phase)
persist_angle = np.repeat(phase_context[:, None], FUTURE_SEGMENTS, axis=1)

probe_sources = {
    "predicted_model": model_angle,
    "teacher_reference": teacher_angle,
    "persist_context": persist_angle,
}
probe_rows = []
for source, angles in probe_sources.items():
    error = circular_error(angles, true_phase)
    probe_rows.append({
        "source": source,
        "mean_angular_error_rad": float(error.mean()),
        "circular_correlation": float(np.cos(error).mean()),
    })
probe_summary = pd.DataFrame(probe_rows)
display(probe_summary)
probe_summary.to_csv(ARTIFACT_DIR / "10_phase_probe.csv", index=False)

probe_scatter_rows = []
for sample_index in range(NORMAL_COUNT):
    for horizon in range(FUTURE_SEGMENTS):
        probe_scatter_rows.append({
            "sequence_id": sequence_ids[sample_index],
            "horizon": horizon + 1,
            "horizon_frames": (horizon + 1) * SEGMENT_LENGTH,
            "true_phase": float(true_phase[sample_index, horizon]),
            "model_predicted_phase": float(model_angle[sample_index, horizon]),
            "teacher_predicted_phase": float(teacher_angle[sample_index, horizon]),
            "persist_phase": float(persist_angle[sample_index, horizon]),
        })
probe_scatter = pd.DataFrame(probe_scatter_rows)
probe_scatter.to_csv(ARTIFACT_DIR / "10_phase_probe_predictions.csv", index=False)
print("saved:", ARTIFACT_DIR / "10_phase_probe.csv")
print("saved:", ARTIFACT_DIR / "10_phase_probe_predictions.csv")
for _, row in probe_summary.iterrows():
    print(
        f"{row['source']:<20s} mean angular error {row['mean_angular_error_rad']:.4f} rad"
    )

## One summary figure

The single artifact figure puts the four measurements side by side: the training curve, the per-horizon forecast loss against the phase-bin baseline, the out-of-distribution comparison, and the latent rollout phase scatter. Smoke runs produce a figure from synthetic fixtures, so read the shapes and code paths, not the values.

In [ ]:
import matplotlib.pyplot as plt

figure, axes = plt.subplots(2, 2, figsize=(13, 9))
history_df.plot(x="epoch", y="loss", marker="o", ax=axes[0, 0], legend=False)
axes[0, 0].set_title("Training: future latent cross-entropy")
axes[0, 0].set_xlabel("epoch")

horizon_frame.plot(
    x="horizon_frames",
    y=["model_future_ce", "baseline_future_ce"],
    marker="o",
    ax=axes[0, 1],
    logy=True,
)
axes[0, 1].set_title("Per-horizon forward loss (log scale)")
axes[0, 1].set_xlabel("frames past the visible window")
axes[0, 1].set_ylabel("latent CE (nats)")

bar_colors = ["#1f77b4"] + ["#d62728"] * (len(condition_summary) - 1)
axes[1, 0].bar(
    condition_summary["condition"],
    condition_summary["mean_forecast_ce"],
    color=bar_colors,
)
axes[1, 0].set_title("Out-of-distribution forecast error by condition")
axes[1, 0].set_ylabel("mean future CE (nats)")
axes[1, 0].tick_params(axis="x", rotation=20)

for horizon in range(FUTURE_SEGMENTS):
    axes[1, 1].scatter(
        true_phase[:, horizon],
        model_angle[:, horizon],
        s=26,
        label=f"{4 * (horizon + 1)} frames",
    )
axes[1, 1].plot([0.0, 2.0 * np.pi], [0.0, 2.0 * np.pi], "k--", linewidth=1)
axes[1, 1].set_title("Latent rollout probe: predicted vs true phase")
axes[1, 1].set_xlabel("true phase (rad)")
axes[1, 1].set_ylabel("predicted phase (rad)")
axes[1, 1].set_xlim(0.0, 2.0 * np.pi)
axes[1, 1].set_ylim(0.0, 2.0 * np.pi)
axes[1, 1].legend(title="horizon", fontsize=8)

plt.tight_layout()
figure_path = ARTIFACT_DIR / "10_world_model_summary.png"
figure.savefig(figure_path, dpi=180, bbox_inches="tight")
print("saved:", figure_path)
plt.show()

## What a real gait world model needs next

This notebook shows that switching S-JEPA from infilling to causal forecasting is a one-line mask change on shared machinery, plus honest evaluation. It also shows how far that switch is from a deployable world model.

- Actions. Walking is closed loop. The next 100 ms of a stride depends on foot contact, balance corrections, and the walker's intention, none of which this model observes. A true gait world model conditions its predictions on an action or command token, even a coarse one such as "continue at this cadence" or "stop".
- Longer horizons and rollouts. Sixteen frames is at most half a stride in these recordings. Multi-stride forecasting needs recursion: feed predicted latents back as context and predict again. Every notebook in the family warns about compounding error, and this one is where that warning becomes concrete.
- Planning. LeCun's framing makes prediction a means to an end: an agent searches over possible futures and picks the one whose imagined outcome scores best. That requires a cost function over latent trajectories, which the tutorials have not yet built.
- Uncertainty. The model returns one deterministic forecast. Real futures branch (which foot will lead, whether the person will stumble), so the next step is a distribution or a small set of sampled futures, not a single latent.
- The teacher limitation. The EMA teacher still sees the full clip, so training measures a supervised forecast against an oracle summary. Deployment would need teacher-free self-consistency, which on twelve clips is very hard to stabilize. Treat this notebook as evidence about the encoder and predictor learning causal dynamics, not about closed-loop stability.
- Evaluation limits. Twelve normal clips from one source video make every forecast claim transductive, and abnormal conditions bring new people, cameras, and pathologies together. Notebook 07's lesson applies: split by source video before claiming that forecast error is a gait biomarker.

The honest summary is short, and it follows the measured tables rather than the hopes. With a fixed future mask, the S-JEPA objective becomes a genuine forecast of future latent states, and it beats a phase-bin mean-latent baseline by more than an order of magnitude, but the per-horizon loss is essentially flat (0.34 to 0.38 across the four horizons), so there is no measured error growth with horizon on the training clips. The out-of-distribution result is the strong one: abnormal clips carry forecast error 7 to 8.6 times the normal training error, although that ratio confounds abnormal dynamics with the input-distribution shift of clips the encoder never saw, and the normal reference is the training set itself, so it is a transductive baseline. The latent rollout phase probe is null: the predicted latents carry no more linear phase information than a persist reference, and the notebook reports that null rather than spinning it. That is a world-model-shaped result with one solid signal (label-free forecast separation) and two honest nulls (horizon growth, rollout phase). Turning it into a planner, with actions and longer autoregressive horizons, is the next experiment.